# 1. Introdução

## 1.1 Identificação do Grupo
* **Grupo:** G19
* **Unidade Curricular:** Inteligência Artificial
* **Elementos:**
  * Afonso Veloso — a28542
  * Rafael Croccia — a28554

---

## 1.2 Contexto e Motivação
Nos últimos anos, tem-se verificado no espaço da União Europeia uma crescente procura por produtos alimentares locais (*Local Food*), impulsionada por motivações de sustentabilidade ambiental, redução da pegada de carbono e valorização da economia regional. As cadeias de abastecimento curtas (*short supply chains*) reduzem as emissões de gases com efeito de estufa associadas ao transporte de longa distância e contribuem para a redução do desperdício de alimentos.

Apesar deste interesse crescente, os pequenos produtores enfrentam desafios operacionais significativos, em particular os elevados custos logísticos e a volatilidade de preços registada no setor hortofrutícola europeu, fatores que pressionam tanto as margens de produção como o poder de compra do consumidor.

Neste enquadramento, a criação de plataformas digitais agregadoras, que articulam produtores, armazéns/centros de recolha (*collection centers*) e consumidores finais, constitui uma solução para tornar a distribuição local viável e sustentável. O desafio computacional central reside no problema do encaminhamento (*routing*): encontrar percursos eficientes de recolha e entrega (*pickup-and-delivery*) que minimizem distâncias e tempos de trânsito, respeitando rigorosamente as restrições logísticas do mundo real, designadamente a capacidade finita das viaturas e as janelas temporais de disponibilidade dos produtos nos produtores e de entrega aos clientes.

---

## 1.3 Objetivo do Projeto
O presente projeto tem por objetivo conceber, formalizar e implementar um **Agente Inteligente de Encaminhamento** capaz de gerar planos de rotas otimizados para uma frota de viaturas de distribuição com cobertura geográfica focada no distrito de Braga e na Póvoa de Varzim.

O problema logístico insere-se na classe dos Problemas de Recolha e Entrega com Janelas Temporais e Capacidade Limitada (*Capacitated Pickup and Delivery Problem with Time Windows* — CPDPTW). Para o resolver e avaliar comparativamente a sua eficácia, são explorados três paradigmas centrais de Inteligência Artificial:

1. **Pesquisa Informada (*Informed Search*):** Implementação dos algoritmos *Greedy Best-First* e *A\**, utilizando duas funções heurísticas distintas baseadas em métricas espaciais e temporais para guiar a exploração no espaço de estados e quantificar o número de nós expandidos;
2. **Pesquisa Local (*Local Search*):** Aplicação do algoritmo *Hill-Climbing* sobre as soluções obtidas previamente pelo método guloso (*Greedy*), explorando operadores de vizinhança para reduzir os custos totais e avaliando a progressão das curvas de custo por iteração;
3. **Problemas de Satisfação de Restrições (*CSP — Constraint Satisfaction Problems*):** Modelação formal do problema através da definição de variáveis, domínios e restrições rígidas (*hard constraints* — precedência de recolha/entrega, capacidade máxima de carga e cumprimento das janelas horárias), incorporando abordagens heurísticas para o tratamento de restrições flexíveis (*soft constraints*) e monitorizando o número de recuos (*backtracks*) necessários para alcançar uma afetação viável.

---

## 1.4 Estrutura do Documento e Entregáveis
O desenvolvimento do projeto encontra-se organizado de forma modular, estruturando-se nos seguintes blocos:

* **Análise Exploratória de Dados (EDA — `01_eda_tiny.ipynb`):** Carregamento dos dados sintéticos via CSV, validação da integridade referencial, análise detalhada das janelas temporais (`availability_start`/`end` e `delivery_start`/`end`), validação de pesos face às capacidades e mapeamento georreferenciado dos pontos de paragem;
* **Modelação do Domínio (`model.py`):** Definição das classes fundamentais (`Problem`, `Request`, `Vehicle`, `State`), implementação das funções de custo operacional e conceção do verificador formal de viabilidade de rotas (*route feasibility checker*), validado através de casos de teste com percursos construídos manualmente (*hand-written routes*);
* **Resolução do Problema (*Problem Solving*):** Execução experimental dos algoritmos de Pesquisa Informada, Pesquisa Local e CSP, apresentando as rotas obtidas para as viaturas da instância de teste (*tiny*);
* **Análise de Resultados e Conclusão:** Comparação quantitativa dos tempos de execução e dos custos operacionais alcançados por cada abordagem, discussão crítica das vantagens e limitações de cada método e levantamento de propostas de desenvolvimento futuro para o agente.


## 2. Pesquisa Informada

Executamos Greedy Best-First Search e A* com as duas heurísticas disponíveis. O limite de expansões evita que a pesquisa consuma recursos indefinidamente na instância completa.

### 2.1 Preparação da instância

Cada estado representa as rotas parciais dos veículos, as localizações atuais, os tempos, a carga e os pedidos recolhidos ou entregues. Os operadores adicionam uma recolha possível ou entregam um pedido já recolhido.

In [4]:
from pathlib import Path

import importlib
import pandas as pd

import ia26_p01_g19.informed_search as informed_search_module
importlib.reload(informed_search_module)

from ia26_p01_g19.informed_search import (
    informed_search_results_to_records,
    run_informed_search_experiments,
)
from ia26_p01_g19.model import Problem

DATA_DIR = Path.cwd() / "src" / "data"
problem = Problem.from_loader(data_dir=DATA_DIR)
print(f"Veículos: {len(problem.vehicles)}")
print(f"Pedidos: {len(problem.requests)}")

Veículos: 2
Pedidos: 23


### 2.2 Execução das quatro combinações

O custo apresentado para uma solução encontrada é recalculado por `Problem.cost_function()`. Se a pesquisa atingir o limite, é usado um fallback construtivo explícito, baseado em inserções viáveis de pares recolha-entrega. A coluna `used_constructive_fallback` distingue esse caso da solução encontrada pela pesquisa pura.

In [5]:
MAX_EXPANSIONS = 5_000

search_results = run_informed_search_experiments(
    problem,
    max_expansions=MAX_EXPANSIONS,
    use_constructive_fallback=True,
)
results_df = pd.DataFrame(
    informed_search_results_to_records(search_results)
)
results_df

,experiment,algorithm,heuristic,found,cost_km,elapsed_seconds,expanded_nodes,generated_nodes,stopped_by_limit,used_constructive_fallback
0,greedy_request_chain,greedy,request_chain,True,1456.19,15.140808,5000,25277,True,True
1,greedy_next_required_location,greedy,next_required_location,True,1456.19,24.123928,5000,48979,True,True
2,a_star_request_chain,a_star,request_chain,True,1456.19,47.756786,5000,110186,True,True
3,a_star_next_required_location,a_star,next_required_location,True,1456.19,49.880599,5000,119709,True,True


### 2.3 Leitura dos resultados

A coluna `found` indica se foi encontrada uma solução. A coluna `stopped_by_limit` indica que a pesquisa terminou por atingir o limite de expansões, não que o problema seja inviável.

### 2.4 Validação das soluções

Para cada solução encontrada, validamos as rotas com `is_route_feasible()`, confirmamos todos os pedidos entregues e recalculamos o custo com `cost_function()`.

In [6]:
def validate_search_solution(problem, result):
    if not result.found or result.solution is None:
        return {
            "valid": False,
            "reason": "Sem solução para validar dentro do limite de expansões.",
            "all_requests_delivered": False,
            "cost_matches": False,
        }

    route_errors = {}
    delivered_request_ids = set()
    for vehicle_id, stops in result.solution.items():
        feasible, reason = problem.is_route_feasible(vehicle_id, stops)
        if not feasible:
            route_errors[vehicle_id] = reason
        delivered_request_ids.update(
            stop.request_id
            for stop in stops
            if stop.action.value == "DELIVERY"
        )

    all_requests_delivered = delivered_request_ids == set(problem.requests)
    recalculated_cost = problem.cost_function(result.solution)
    cost_matches = result.cost is not None and abs(
        result.cost - recalculated_cost
    ) < 1e-9
    return {
        "valid": not route_errors and all_requests_delivered and cost_matches,
        "route_errors": route_errors,
        "all_requests_delivered": all_requests_delivered,
        "delivered_requests": len(delivered_request_ids),
        "expected_requests": len(problem.requests),
        "reported_cost_km": result.cost,
        "recalculated_cost_km": recalculated_cost,
        "cost_matches": cost_matches,
    }

validation_records = [
    {"experiment": name, **validate_search_solution(problem, result)}
    for name, result in search_results.items()
]
validation_df = pd.DataFrame(validation_records)
validation_df

,experiment,valid,route_errors,all_requests_delivered,delivered_requests,expected_requests,reported_cost_km,recalculated_cost_km,cost_matches
0,greedy_request_chain,True,{},True,23,23,1456.19,1456.19,True
1,greedy_next_required_location,True,{},True,23,23,1456.19,1456.19,True
2,a_star_request_chain,True,{},True,23,23,1456.19,1456.19,True
3,a_star_next_required_location,True,{},True,23,23,1456.19,1456.19,True


### 2.5 Rotas obtidas

A rota construtiva validada é apresentada por veículo. Cada linha identifica o pedido, a ação e a localização da paragem.

In [7]:
reference_result = next(
    result for result in search_results.values()
    if result.solution is not None
)

route_records = []
for vehicle_id, stops in reference_result.solution.items():
    for position, stop in enumerate(stops, start=1):
        route_records.append(
            {
                "vehicle_id": vehicle_id,
                "position": position,
                "request_id": stop.request_id,
                "action": stop.action.value,
                "location_id": stop.location_id,
            }
        )
routes_df = pd.DataFrame(route_records)
route_costs_df = pd.DataFrame(
    [
        {
            "vehicle_id": vehicle_id,
            "stops": len(stops),
            "cost_km": problem.route_cost(vehicle_id, stops),
        }
        for vehicle_id, stops in reference_result.solution.items()
    ]
)
display(route_costs_df)
display(routes_df)

,vehicle_id,stops,cost_km
0,1,24,776.46
1,2,22,679.73


,vehicle_id,position,request_id,action,location_id
0,1,1,3,PICKUP,2
1,1,2,3,DELIVERY,12
2,1,3,13,PICKUP,3
3,1,4,14,PICKUP,1
4,1,5,23,PICKUP,5
5,1,6,14,DELIVERY,11
6,1,7,23,DELIVERY,8
7,1,8,13,DELIVERY,7
8,1,9,9,PICKUP,3
9,1,10,9,DELIVERY,8


### 2.6 Análise crítica

A comparação deve considerar o tempo, os nós gerados e o uso do fallback. O custo é igual nos quatro resultados porque todos usam a mesma solução construtiva quando a pesquisa atinge o limite.

In [8]:
analysis = {
    "fastest_experiment": results_df.loc[
        results_df["elapsed_seconds"].idxmin(), "experiment"
    ],
    "fewest_generated_nodes": results_df.loc[
        results_df["generated_nodes"].idxmin(), "experiment"
    ],
    "best_cost_km": results_df["cost_km"].min(),
    "all_used_fallback": bool(
        results_df["used_constructive_fallback"].all()
    ),
}
analysis

{'fastest_experiment': 'greedy_request_chain',
 'fewest_generated_nodes': 'greedy_request_chain',
 'best_cost_km': np.float64(1456.19),
 'all_used_fallback': True}

### 2.7 Interpretação do fallback construtivo

O fallback não substitui a pesquisa informada. A pesquisa continua a ser executada e contabiliza os nós expandidos e gerados até ao limite. Quando esse limite é atingido, o construtor percorre os pedidos pela ordem dos prazos de recolha e testa inserções de pares `PICKUP -> DELIVERY` em todas as posições possíveis das rotas. Apenas inserções aprovadas por `is_route_feasible()` são consideradas.

Assim, `found=True` juntamente com `stopped_by_limit=True` e `used_constructive_fallback=True` significa que foi obtida uma solução viável, mas não pela pesquisa pura dentro do limite.

## 3. Pesquisa Local: Hill-Climbing

O Hill-Climbing começa com uma solução construtiva viável e procura melhorias locais. São usados dois operadores: (1) realocação de um pedido completo, reinscrevendo o par `PICKUP -> DELIVERY` noutra posição e/ou noutro veículo; (2) troca das identidades de dois pedidos. Apenas rotas que continuam viáveis são consideradas.

In [ ]:
from ia26_p01_g19.hill_climbing import hill_climbing

hill_result = hill_climbing(
    problem,
    initial_solution=reference_result.solution,
    max_iterations=10,
)

hill_summary = pd.DataFrame(
    [
        {
            "initial_cost_km": hill_result.cost_history[0],
            "final_cost_km": hill_result.cost,
            "iterations": hill_result.iterations,
            "generated_neighbors": hill_result.generated_neighbors,
            "elapsed_seconds": hill_result.elapsed_seconds,
            "converged": hill_result.converged,
        }
    ]
)
hill_summary

In [ ]:
import matplotlib.pyplot as plt

plt.plot(
    range(len(hill_result.cost_history)),
    hill_result.cost_history,
    marker="o",
)
plt.xlabel("Iteração")
plt.ylabel("Custo (km)")
plt.title("Hill-Climbing: custo por iteração")
plt.grid(True)
plt.show()

O algoritmo aceita apenas melhorias estritas. Quando nenhum vizinho viável reduz o custo, o algoritmo atinge um ótimo local e termina. O custo final é recalculado com `cost_function()` e as rotas são validadas com `is_route_feasible()`.

### 3.1 Validação da solução local

A solução final é validada de forma independente: cada rota deve respeitar as restrições do modelo, os 23 pedidos devem aparecer numa entrega e o custo deve coincidir com `cost_function()`.

In [ ]:
def validate_hill_climbing_solution(problem, result):
    route_errors = {}
    delivered_request_ids = set()
    for vehicle_id, stops in result.solution.items():
        feasible, reason = problem.is_route_feasible(vehicle_id, stops)
        if not feasible:
            route_errors[vehicle_id] = reason
        delivered_request_ids.update(
            stop.request_id
            for stop in stops
            if stop.action.value == "DELIVERY"
        )

    recalculated_cost = problem.cost_function(result.solution)
    all_requests_delivered = delivered_request_ids == set(problem.requests)
    cost_matches = abs(result.cost - recalculated_cost) < 1e-9
    return {
        "valid": not route_errors and all_requests_delivered and cost_matches,
        "route_errors": route_errors,
        "delivered_requests": len(delivered_request_ids),
        "expected_requests": len(problem.requests),
        "reported_cost_km": result.cost,
        "recalculated_cost_km": recalculated_cost,
        "cost_matches": cost_matches,
    }

hill_validation = validate_hill_climbing_solution(problem, hill_result)
pd.DataFrame([hill_validation])

### 3.2 Rotas finais

A tabela seguinte mostra a solução depois da melhoria local, incluindo a distância de cada veículo.

In [ ]:
hill_route_costs = pd.DataFrame(
    [
        {
            "vehicle_id": vehicle_id,
            "stops": len(stops),
            "cost_km": problem.route_cost(vehicle_id, stops),
        }
        for vehicle_id, stops in hill_result.solution.items()
    ]
)
hill_route_costs

### 3.3 Análise crítica

O custo deve diminuir monotonamente porque o algoritmo aceita apenas vizinhos estritamente melhores. Isto demonstra melhoria local, mas não prova optimalidade global. O resultado depende da solução inicial e pode ficar preso num ótimo local.